In [1]:
import polars as pl
import re

# simple.csv comes out of extract_glossary.py (step 0) and already includes
# the additional abbreviations that used to live in my_additions.csv
simple = pl.read_csv("data/simple.csv")
dioceses = pl.read_csv("data/dioceses.csv").rename({"abbreviation": "Abkürzung", "expansion": "Auflösung"})

# diagonal concat fills the columns missing from dioceses with null;
# since their abbreviations are unique and have no RG1-RG9 entries, the consistency
# cell below marks them as applying to all volumes
simple = pl.concat([simple, dioceses], how="diagonal")

simple = simple.with_columns(pl.col("Abkürzung").str.split(' ').list.len().alias("abbr_parts"))
simple = simple.sort(by="abbr_parts", descending=True)

simple

Abkürzung,Auflösung,Übersetzung,Anmerkungen,Wortstamm,Deklination,RG1,RG2,RG3,RG4,RG5,RG6,RG7,RG8,RG9,Bemerkungen_1,Bemerkungen_2,volumes,abbr_parts
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,u32
"""s. e. s. o. n. c.""","""sed eius superveniente obitu n…",null,null,null,null,null,null,null,null,null,"""s. e. s. o. n. c.""",null,null,null,null,null,"""6""",6
"""ben. c. v. s. c.""","""beneficium cum vel sine cura""","""Pfründe mit oder ohne Seelsorg…","""siehe benef. c. v. s. c.""",null,null,"""ben. c. v. s. c.""",null,null,null,null,null,null,null,null,null,null,"""1""",5
"""benef. c. v. s. c.""","""beneficium cum vel sine cura""","""Pfründe mit oder ohne Seelsorg…",null,null,null,null,null,null,null,null,"""benef. c. v. s. c.""",null,null,null,null,null,"""1|6""",5
"""de conf. disp. sup. matrim.""","""confirmatione dispensationis s…",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""4|6|8""",5
"""de ref. disp. sup. matrim.""","""de reformatione dispensationis…",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""5""",5
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Wien.""","""Viennensis""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1
"""Wladislav.""","""Wladislaviensis""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1
"""Wormat.""","""Wormatiensis""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1


making things more consistent (by assuming that if a row has no entries in RG1-RG9 and there are no other rows with that abbreviation, it can be applied to all volumes)

In [2]:
simple = simple.with_columns(
    pl.when((~pl.col("Abkürzung").is_duplicated()) & pl.all_horizontal(pl.col("^RG[1-9]$").is_null())).then(pl.col("Abkürzung")).otherwise(pl.col("RG1")).alias("RG1"),
    pl.when((~pl.col("Abkürzung").is_duplicated()) & pl.all_horizontal(pl.col("^RG[1-9]$").is_null())).then(pl.col("Abkürzung")).otherwise(pl.col("RG2")).alias("RG2"),
    pl.when((~pl.col("Abkürzung").is_duplicated()) & pl.all_horizontal(pl.col("^RG[1-9]$").is_null())).then(pl.col("Abkürzung")).otherwise(pl.col("RG3")).alias("RG3"),
    pl.when((~pl.col("Abkürzung").is_duplicated()) & pl.all_horizontal(pl.col("^RG[1-9]$").is_null())).then(pl.col("Abkürzung")).otherwise(pl.col("RG4")).alias("RG4"),
    pl.when((~pl.col("Abkürzung").is_duplicated()) & pl.all_horizontal(pl.col("^RG[1-9]$").is_null())).then(pl.col("Abkürzung")).otherwise(pl.col("RG5")).alias("RG5"),
    pl.when((~pl.col("Abkürzung").is_duplicated()) & pl.all_horizontal(pl.col("^RG[1-9]$").is_null())).then(pl.col("Abkürzung")).otherwise(pl.col("RG6")).alias("RG6"),
    pl.when((~pl.col("Abkürzung").is_duplicated()) & pl.all_horizontal(pl.col("^RG[1-9]$").is_null())).then(pl.col("Abkürzung")).otherwise(pl.col("RG7")).alias("RG7"),
    pl.when((~pl.col("Abkürzung").is_duplicated()) & pl.all_horizontal(pl.col("^RG[1-9]$").is_null())).then(pl.col("Abkürzung")).otherwise(pl.col("RG8")).alias("RG8"),
    pl.when((~pl.col("Abkürzung").is_duplicated()) & pl.all_horizontal(pl.col("^RG[1-9]$").is_null())).then(pl.col("Abkürzung")).otherwise(pl.col("RG9")).alias("RG9"),
)

creating a dictionary with the expansions for each volume

In [3]:
#volume_specific = [simple.filter(pl.col("volumes").str.contains(rf"{volume}|\*")) for volume in range(10)]
volume_specific_unique = {}
volume_specific_duplicates = {}

for volume in range(1,10):
    volume_specific_unique[volume] = simple.filter(pl.col(rf"^RG{volume}$").str.contains(pl.col("Abkürzung")))
    volume_specific_unique[volume] = volume_specific_unique[volume].filter(~ pl.col("Abkürzung").is_duplicated())
    
    volume_specific_duplicates[volume] = simple.filter(pl.col(rf"^RG{volume}$").str.contains(pl.col("Abkürzung")))
    volume_specific_duplicates[volume] = volume_specific_duplicates[volume].filter(pl.col("Abkürzung").is_duplicated())

In [4]:
rg = pl.read_csv("data/RG_header_sublemma_all.csv").select(["volume", "nr_RG", "nr_suffix", "header_no_tags", "regest_no_tags", "id_RG_all"])
rg = rg.sort(by=["volume", "nr_RG", "nr_suffix"])

In [5]:
def expand_rg_simple(rg_texts, volume_specific_expansions):
    volumes = {}
    for volume in range(1,10):
        expansions = volume_specific_expansions[volume]
        texts = rg_texts.filter(pl.col("volume") == volume)
        
        for row in expansions.iter_rows(named=True):
            # TODO
            # since all abbreviations, but no expansions contain `.`, we don't need to worry about expanding what has already been expanded

            # this is what I do right now: replace_all(r"\b(eccl\.)", "ecclesia")
            # the following line would also get ones where the dot is missing or there is instead, e.g. a comma, but for words where there could be a mix-up (if e.g. "eccl" was also a word), this would be problematic
            #replace_all(r"\b(eccl\.)", "ecclesia").str.replace_all(r"\b(eccl)\b", "ecclesia")

            texts = texts.with_columns(pl.col("header_no_tags").str.replace_all(r'\b(' + re.escape(row["Abkürzung"]).replace(r"\ ", r"[\ \t]*") + ')', row["Auflösung"]).alias("header_no_tags"))
            texts = texts.with_columns(pl.col("regest_no_tags").str.replace_all(r'\b(' + re.escape(row["Abkürzung"]).replace(r"\ ", r"[\ \t]*") + ')', row["Auflösung"]).alias("regest_no_tags"))

            if row["Abkürzung"] == "eccl.":
                if volume == 5:
                    print(texts.filter(pl.col("id_RG_all") == id).get_column("regest_no_tags").item())

        volumes[volume] = texts
    
    return pl.concat(volumes.values())

In [6]:
expanded_texts = expand_rg_simple(rg, volume_specific_unique)

In [7]:
id = "10906306-3"
print(rg.filter(pl.col("id_RG_all") == id).get_column("regest_no_tags").item())
print(expanded_texts.filter(pl.col("id_RG_all") == id).get_column("regest_no_tags").item())

Dom. o. fr. min. e.m. op. Z. Traiect. dioc. de observ. nunc. ap. auct. ad instantiam genitoris Adolphi Gelrie et Juliacen. [ducis] et comitis Zutphanien. erecta sed adhuc non confecta: supplic. d. Adolpho duce de lic. dom. in d. op. de novo erigendi et fratres ad illam transferendi 16. mai. 1470 S 658 282rs.
Dom. o. frater min. extra muros opidum Z. Traiectensis diocesis de observ. nuncupatus ap. auctoritas ad instantiam genitoris Adolphi Gelrie et Juliacen. [ducis] et comitis Zutphanien. erecta sed adhuc non confecta: supplic. d. Adolpho duce de lic. dom. in d. opidum de novo erigendi et fratres ad illam transferendi 16. mai. 1470 S 658 282rs.


approximating the number of abbreviations by the number of `.`

In [8]:
# filtering for all volumes except 10, because we don't have any rules for volumes 10 yet and are dropping this one when expanding
counts_rg = rg.filter(pl.col("volume") != 10).with_columns(pl.col("header_no_tags").str.count_matches(r"\w+\.").sum().alias("countH"), pl.col("regest_no_tags").str.count_matches(r"\.").sum().alias("countR"))
abbreviations_rg = counts_rg.row(0)[-2] + counts_rg.row(0)[-1]

counts_expanded = expanded_texts.filter(pl.col("volume") != 10).with_columns(pl.col("header_no_tags").str.count_matches(r"\w+\.").sum().alias("countH"), pl.col("regest_no_tags").str.count_matches(r"\.").sum().alias("countR"))
abbreviations_expanded = counts_expanded.row(0)[-2] + counts_expanded.row(0)[-1]

print(f"abbreviations in rg: {abbreviations_rg}")
print(f"abbreviations after expanding: {abbreviations_expanded}")

abbreviations in rg: 2482704
abbreviations after expanding: 1062702


- without my additions: 1.427 million abbreviations
- with: 1.226 million
- with interpreting the ones without RG1-RG9 as applying to all volumes (also when duplicated): +10k - most of them are not meant to apply to all volumes anyway I think 

In [9]:
def find_overlapping_matches(text: str, pattern: str) -> list[str]:
    """Find all overlapping matches using lookahead."""
    if text is None:
        return []
    # Wrap pattern in a lookahead to allow overlapping matches
    lookahead_pattern = f"(?=({pattern}))"
    return [m.group(1) for m in re.finditer(lookahead_pattern, text)]

In [10]:
def find_abbreviations(texts: pl.DataFrame) -> pl.DataFrame:
    left_over = texts.with_columns(
        pl.col("header_no_tags").str.extract_all(r"\b(\w+\.)").alias("1h"),
        pl.col("regest_no_tags").str.extract_all(r"\b(\w+\.)").alias("1r"),    
        pl.col("header_no_tags").map_elements(
            lambda x: find_overlapping_matches(x, r"\b(\w+\.\ ?\w+\.)"),
            return_dtype=pl.List(pl.String)
        ).alias("2h"),
        pl.col("regest_no_tags").map_elements(
            lambda x: find_overlapping_matches(x, r"\b(\w+\.\ ?\w+\.)"),
            return_dtype=pl.List(pl.String)
        ).alias("2r"),
        pl.col("header_no_tags").map_elements(
            lambda x: find_overlapping_matches(x, r"\b(\w+\.\ ?\w+\.\ ?\w+\.)"),
            return_dtype=pl.List(pl.String)
        ).alias("3h"),
        pl.col("regest_no_tags").map_elements(
            lambda x: find_overlapping_matches(x, r"\b(\w+\.\ ?\w+\.\ ?\w+\.)"),
            return_dtype=pl.List(pl.String)
        ).alias("3r")
        #pl.col("header_no_tags").str.extract_all(r"\b(\w+\.\ ?\w+\.)").alias("2h"),
        #pl.col("regest_no_tags").str.extract_all(r"\b(\w+\.\ ?\w+\.)").alias("2r"),
        #pl.col("header_no_tags").str.extract_all(r"\b(\w+\.\ ?\w+\.\ ?\w+\.)").alias("3h"),
        #pl.col("regest_no_tags").str.extract_all(r"\b(\w+\.\ ?\w+\.\ ?\w+\.)").alias("3r")
    )
    abbreviations = pl.concat((
        left_over.get_column("1h").explode().drop_nulls(),
        left_over.get_column("1r").explode().drop_nulls(),
        left_over.get_column("2h").explode().drop_nulls(),
        left_over.get_column("2r").explode().drop_nulls(),
        left_over.get_column("3h").explode().drop_nulls(),
        left_over.get_column("3r").explode().drop_nulls())
        ).value_counts().rename({"1h": "abbreviation"}).with_columns(pl.col("abbreviation").str.split(' ').list.len().alias("parts")).sort("count", descending=True)

    return abbreviations

original = find_abbreviations(rg.filter(pl.col("volume") != 10))
left_over = find_abbreviations(expanded_texts.filter(pl.col("volume") != 10))

/tmp/ipykernel_23741/2675830413.py:27: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  left_over.get_column("1h").explode().drop_nulls(),
/tmp/ipykernel_23741/2675830413.py:28: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  left_over.get_column("1r").explode().drop_nulls(),
/tmp/ipykernel_23741/2675830413.py:29: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  left_over.get_column("2h").explode().drop_nulls(),
/tmp/ipykernel_23741/2675830413.py:30: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  left_over.get_c

count of abbreviations in RG before and after expanding

In [11]:
from helper_functions import side_by_side

side_by_side(
    original.head(20), left_over.head(20),
    None, # this adds spacing
    original.filter(pl.col("parts") > 1).head(20), left_over.filter(pl.col("parts") > 1).head(20)
    )

,abbreviation,count,parts
0,eccl.,157073,1
1,dioc.,128830,1
2,s.,107232,1
3,p.,82106,1
4,o.,79070,1
5,m.,59572,1
6,par.,57804,1
7,par. eccl.,56427,2
8,can.,54203,1
9,d.,52053,1


potential for abbreviations in simple.csv/complex.csv (e.g. if `can.` were to have a clear expansion for all volumes, the gain would be 50k expanded abbreviations)

In [12]:
complex = pl.read_csv("data/complex.csv")
potential_complex = left_over.rename({"abbreviation": "Abkürzung"}).join(complex, on="Abkürzung", how="inner").select("Abkürzung", "count", "parts").unique().sort(by="count",descending=True) # the join is just to filter for the ones that are in complex.csv
potential_simple = left_over.rename({"abbreviation": "Abkürzung"}).join(simple, on="Abkürzung", how="inner").select("Abkürzung", "count", "parts").unique().sort(by="count",descending=True) # the join is just to filter for the ones that are in simple.csv

side_by_side(
    potential_complex.head(20), potential_simple.head(20)
    )

,Abkürzung,count,parts
0,can.,54201,1
1,d.,52049,1
2,s.,40095,1
3,p.,34077,1
4,cler.,31605,1
5,sup.,29981,1
6,prov.,28909,1
7,o.,24735,1
8,b.,20772,1
9,mai.,16844,1


In [ ]:
expanded_texts.write_csv("data/once_expanded.csv")